### [ 주제 : 결과 안정성 검증 — 여러 seed로 AUC 평균±편차 ]

AUC 한 번 값(0.94~0.96)은 실행마다 흔들린다 (모델 초기값·데이터 분리가 랜덤).
여러 seed로 돌려 **평균 ± 표준편차**로 보고하면 훨씬 믿음직한 수치가 된다.

또한 이 실험은 앞서 만든 **`src/` 모듈을 import해서** 파이프라인을 재사용한다 (복붙 없음).

In [2]:
## [0] 준비 : src 모듈 불러오기
import sys
sys.path.insert(0, '..')          # 노트북은 notebooks/에서 실행 → 프로젝트 루트를 경로에 추가
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import numpy as np
import torch
from src import config, data, train, evaluate

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('학습 장치 :', DEVICE)

학습 장치 : cuda


In [3]:
## [1] 여러 seed로 파이프라인 반복 (레벨당 학습 1~2분 → 5개면 5~10분)
# TODO:
#  SEEDS = [0, 1, 2, 3, 4]
#  aucs = []
#  for seed in SEEDS:
#      torch.manual_seed(seed)                                   # 모델 초기값 고정
#      trainNP, teN, teA = data.load_split_normalize(seed=seed)  # 분리도 seed로
#      model, _ = train.train_autoencoder(trainNP, DEVICE)
#      en = evaluate.recon_errors(model, teN, DEVICE)
#      ea = evaluate.recon_errors(model, teA, DEVICE)
#      auc = evaluate.compute_auc(en, ea)
#      aucs.append(auc)
#      print(f'seed {seed}: AUC {auc:.4f}')
SEEDS = [0, 1, 2, 3, 4]
aucs = []
for seed in SEEDS:
    torch.manual_seed(seed)
    trainNP, teN, teA = data.load_split_normalize(seed=seed)
    model, _ = train.train_autoencoder(trainNP, DEVICE)
    en = evaluate.recon_errors(model,teN,DEVICE)
    ea = evaluate.recon_errors(model,teA,DEVICE)
    auc = evaluate.compute_auc(en, ea)
    aucs.append(auc)
    print(f'seed {seed}: AUC {auc:.4f}')

seed 0: AUC 0.9622
seed 1: AUC 0.9494
seed 2: AUC 0.9825
seed 3: AUC 0.9604
seed 4: AUC 0.9737


In [4]:
## [2] 평균 ± 표준편차
# TODO:

aucs = np.array(aucs)

print(f'AUC 평균 {aucs.mean():.4f} ± {aucs.std():.4f}  (seed {len(aucs)}개)')



AUC 평균 0.9656 ± 0.0114  (seed 5개)


In [ ]:
## [3] seed별 AUC 변동 시각화
# 평균만 보면 seed에 따른 흔들림이 가려지므로 각 실행값을 점과 선으로 표시한다.
import matplotlib.pyplot as plt
import koreanize_matplotlib

AUC_MEAN = aucs.mean()
AUC_STD = aucs.std()

plt.figure(figsize=(8, 4))
plt.plot(SEEDS, aucs, marker='o', linewidth=2, label='Linear AE AUC')
plt.axhline(AUC_MEAN, color='tab:red', linestyle='--', label=f'평균 {AUC_MEAN:.4f}')
plt.axhspan(AUC_MEAN-AUC_STD, AUC_MEAN+AUC_STD, color='tab:red', alpha=0.15, label='평균 ± 표준편차')
plt.xticks(SEEDS)
plt.ylim(0.90, 1.00)
plt.xlabel('시드')
plt.ylabel('AUC')
plt.title('시드별 Linear AE 성능 안정성')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.show()


## 결과 (실행 후 채우기)

- AUC = 0.9x ± 0.0x (seed 5개)
- 편차가 작으면 → "한 번의 운이 아니라 안정적인 성능"이라는 근거
- 이렇게 보고하면 "AUC 0.95" 한 숫자보다 신뢰가 높음